## 1. Install dependencies

# Study & Clipboard Agent

Run from the `agent-project` folder with Python 3.11+. Restart the kernel after the first install if needed. No API keys are included.

In [ ]:
%pip install -r requirements.txt

## 2. Imports

The requested agent factory and executor are in `langchain-classic` on LangChain 1.x. All decorated tools live in `tools.py`.

In [ ]:
import os
import subprocess
import sys
from uuid import uuid4

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_groq import ChatGroq

from config import DEFAULT_MODEL, PROJECT_DIR, get_llm, load_environment, missing_credentials
from db import init_db
from models import Flashcard, FlashcardSet

## 3. Load .env and enable LangSmith

Copy `.env.example` to `.env` and enter both keys before running live conversations. Tracing is enabled for Groq, agent, and nested tool chains. Without credentials, the notebook explicitly skips LLM calls and still runs the local tools and real currency requests.

In [ ]:
load_environment()  # Loads agent-project/.env via python-dotenv.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_TRACING"] = "true"
missing = missing_credentials()
credentials_ready = not missing
live_http_enabled = os.getenv("RUN_LIVE_HTTP", "true").strip().lower() == "true"
print("LangSmith tracing:", os.environ["LANGCHAIN_TRACING_V2"])
print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])
print("Credentials: ready" if credentials_ready else "Missing credentials: " + ", ".join(missing))

## 4. Initialize SQLite

`notes.db` is stored beside the Python modules, independent of the kernel's working directory. Each operation commits and closes its connection.

In [ ]:
init_db()
print("SQLite initialized: notes.db")

Default model: **Groq (`openai/gpt-oss-120b`)**. `get_llm()` uses `DEFAULT_MODEL`; `LLM_MODEL` in `.env` can override it.

In [ ]:
from config import get_llm

llm: ChatGroq | None = None
if credentials_ready:
    llm = get_llm()  # Groq openai/gpt-oss-120b, configured in config.py
    print("Groq initialized:", os.getenv("LLM_MODEL") or DEFAULT_MODEL)
else:
    print("Groq initialization skipped: add both API keys to .env.")

## 6. Import the nine tools

All tools have typed arguments, docstrings, guarded implementations, and validation-error handling. Translation, explanation, and grammar use `prompt | llm | parser`; flashcards use `prompt | llm.with_structured_output(FlashcardSet)`.

In [ ]:
from tools import ALL_TOOLS, convert_currency, detect_language, fix_layout, save_note, search_notes

tools = ALL_TOOLS
assert len(tools) == 9
print("Tools:", ", ".join(tool.name for tool in tools))
print("Layout check:", fix_layout.invoke({"text": "руддщ"}))
print("Reverse layout check:", fix_layout.invoke({"text": "ghbdtn"}))
print("Language check:", detect_language.invoke({"text": "Қазақ тілі"}))

## 7. Build the prompt

`chat_history` carries prior user/assistant messages. `agent_scratchpad` carries this turn's tool calls and observations. The first system message is the exact requested prompt.

In [ ]:
SYSTEM_PROMPT = "You are Study & Clipboard Agent. You have these tools: fix_layout, detect_language, translate_text, explain_selection, grammar_check, make_flashcards, save_note, search_notes, convert_currency. Use them when relevant. Be concise."

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])
print("Prompt ready with chat_history and agent_scratchpad placeholders.")

## 8. Create the tool-calling agent

Groq selects a tool and its arguments. AgentExecutor executes the tool, feeds back its observation, and asks Groq for the final response. Intermediate steps make tool selection visible; the loop is bounded.

In [ ]:
agent = None
agent_executor = None
if llm is not None:
    agent = create_tool_calling_agent(llm, tools, prompt)
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        return_intermediate_steps=True,
        handle_parsing_errors=True,
        max_iterations=6,
        max_execution_time=180,
    )
    print("Tool-calling agent ready.")
else:
    print("Agent setup skipped: Groq credentials are missing.")

## 9. Add session memory

The same session ID reuses ChatMessageHistory. Different IDs isolate conversations. This chat memory is in RAM; notes persist separately in SQLite across restarts.

The executor's final content blocks are converted to plain text before entering chat history. This prevents signature-bearing text blocks from being interpreted as separate messages; tool-call signatures inside the executor are preserved.

In [ ]:
def normalize_agent_output(result: dict) -> dict:
    """Convert final content blocks to text before the history wrapper sees them.

    The LLM can return text blocks with signature metadata. Those blocks are
    content inside one AI message, not a sequence of chat messages. Internal
    tool-call messages and their signatures remain untouched.
    """
    output = result["output"]
    if isinstance(output, (list, BaseMessage)):
        message = output if isinstance(output, BaseMessage) else AIMessage(content=output)
        return {**result, "output": StrOutputParser().invoke(message)}
    return result


store: dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

agent_with_history = None
if agent_executor is not None:
    agent_with_history = RunnableWithMessageHistory(
        agent_executor | RunnableLambda(normalize_agent_output),
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="output",
    )
print("Session history ready.")

## 10. Example conversations

Three requested conversations are below, plus a supported USD→EUR pair. Without keys, **direct tool checks are labeled** and the multi-turn LLM example is skipped; those are not fabricated agent conversations. ExchangeRate API supports both KZT and EUR. Rates are the latest published rates, not tick-by-tick market quotes.

In [ ]:
import re

def is_limit_error(exc: Exception) -> bool:
    status = getattr(exc, "status_code", None)
    response_status = getattr(getattr(exc, "response", None), "status_code", None)
    message = f"{type(exc).__name__} {exc}".lower()
    return (
        status in (413, 429) or response_status in (413, 429)
        or "rate_limit" in message or "rate limit" in message
        or "ratelimit" in message or bool(re.search(r"\b(?:413|429)\b", message))
    )


def run_example(message: str, session_id: str, direct_tool=None, arguments=None):
    print("User:", message)
    if agent_with_history is None:
        if direct_tool is convert_currency and not live_http_enabled:
            print("Skipped live currency request: RUN_LIVE_HTTP=false.")
        elif direct_tool is not None:
            print("Direct tool check (agent skipped):", direct_tool.invoke(arguments))
        else:
            print("Skipped live agent response: add API keys and select an available Groq model.")
        return None
    try:
        result = agent_with_history.invoke(
            {"input": message},
            config={"configurable": {"session_id": session_id}},
        )
        for action, observation in result.get("intermediate_steps", []):
            print(f"Tool: {action.tool} | Arguments: {action.tool_input}")
            print("Tool result:", observation)
        print("Assistant:", result["output"])
        return result
    except Exception as exc:
        if is_limit_error(exc):
            print("Text too long, please shorten it")
            return None
        if credentials_ready:
            raise  # A live verification failure must be visible, not silently skipped.
        # A retired model, invalid key, or exhausted quota should not stop local demos.
        print(f"Live agent call failed ({type(exc).__name__}). Check model availability, keys, and quota.")
        if direct_tool is convert_currency and not live_http_enabled:
            print("Skipped live currency request: RUN_LIVE_HTTP=false.")
        elif direct_tool is not None:
            print("Direct tool check (agent failed):", direct_tool.invoke(arguments))
        return None

print("=== Example 1: single turn ===")
run_example("руддщ", "layout-demo", fix_layout, {"text": "руддщ"})

print("\n=== Example 2: multi-turn memory ===")
study_session = "study-" + uuid4().hex[:8]
first = run_example("Explain backpropagation", study_session)
second = run_example("make flashcards from that", study_session)
if first is not None and second is not None:
    messages = get_session_history(study_session).messages
    assert len(messages) == 4, "Two user/assistant turns must be remembered."
    assert messages[1].content == first["output"]
    assert all(isinstance(message, BaseMessage) for message in messages)
    assert any(action.tool == "explain_selection" for action, _ in first["intermediate_steps"])
    flashcard_calls = [action for action, _ in second["intermediate_steps"] if action.tool == "make_flashcards"]
    assert flashcard_calls, "Follow-up should call make_flashcards using the previous explanation."
    assert flashcard_calls[0].tool_input.get("text", "").strip()
    print("Memory check: 4 messages retained; follow-up called make_flashcards.")

print("\n=== Example 3: real API, requested KZT pair ===")
run_example("Convert 100 USD to KZT", "currency-kzt", convert_currency,
            {"amount": 100.0, "from_currency": "USD", "to_currency": "KZT"})

print("\n=== Additional supported live pair ===")
run_example("Convert 100 USD to EUR", "currency-eur", convert_currency,
            {"amount": 100.0, "from_currency": "USD", "to_currency": "EUR"})

## 11. Prove notes persist

Save through the tool, then open SQLite in a **separate Python process** and search again through the tool. Re-running creates another timestamped demo note; the unique tag identifies this run. Restart the kernel and search an earlier printed tag to retrieve the same note.

In [ ]:
note_tag = "persistence-" + uuid4().hex[:8]
print(save_note.invoke({
    "content": "Backpropagation computes gradients with the chain rule to train neural networks.",
    "tags": "study,backpropagation," + note_tag,
}))

proof = subprocess.run(
    [sys.executable, "-c",
     "import sys; import db; rows = db.search_notes(sys.argv[1]); "
     "assert rows, 'Saved note was not found'; "
     "print('Persistence check: saved note reopened in a separate Python process.')",
     note_tag],
    cwd=PROJECT_DIR,
    check=True,
    capture_output=True,
    text=True,
)
print(proof.stdout.strip())
print(search_notes.invoke({"query": note_tag}))
print("Re-run search_notes later with tag:", note_tag)

In [ ]:
try:
    agent_with_history.invoke(
        {"input": "Explain the chain rule in calculus."},
        config={"configurable": {"session_id": "my-session"}},
    )
except Exception as exc:
    if is_limit_error(exc):
        print("Text too long, please shorten it")
    else:
        raise
